# K-mer Feature Pipeline (Binary presence)

This notebook performs k-mer feature selection and trains models using binary presence/absence
features for selection and final modeling. The prevalence-first approach keeps memory use low.
Models and vocabulary are saved to
`output/.`

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split


## Helper functions
These helpers iterate per-genome k-mer dump files, build prevalence counts, construct
sparse matrices (binary mode), and load phenotype labels.

In [2]:
# Helper utilities for k-mer pipelines (binary-presence notebook)
# Each function below includes a short comment/docstring explaining its role, inputs, and outputs.
from typing import Iterable

def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
    """Read a per-genome k-mer dump and return a dict of {kmer: count}.

    Parameters:
    - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

    Returns:
    - dict mapping kmer (str) to integer count.
    """
    kmers: dict[str, int] = {}
    with dump_path.open("r", encoding="utf8", errors="ignore") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            kmer, cnt = parts
            try:
                kmers[kmer] = int(cnt)
            except ValueError:
                continue
    return kmers


def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
    """Aggregate presence counts for each k-mer across a list of genomes.

    Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

    Parameters:
    - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
    - genome_ids: list of genome identifiers to include.

    Returns:
    - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
    """
    prevalence: Counter = Counter()
    dump_dir = Path(dump_dir)
    for gid in genome_ids:
        dump_path = dump_dir / f"{gid}_db_kmers.txt"
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        prevalence.update(kmers.keys())
    return prevalence


def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
                               min_frac: float = 0.02, max_frac: float = 0.95) -> list[str]:
    """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

    Parameters:
    - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
    - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
    - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
    - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

    Returns:
    - List of k-mer strings that pass the prevalence filters.
    """
    min_count = int(np.ceil(min_frac * n_genomes))
    max_count = int(np.floor(max_frac * n_genomes))
    vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
    return vocab


def build_sparse_matrix(
    dump_dir: str,
    genome_ids: list[str],
    vocab: list[str],
    binary: bool = True,
    chunk_size: int = 100,
) -> sparse.csr_matrix:
    """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

    This reads each genome's dump and populates the matrix using the provided `vocab` index.
    When `binary` is True, presence is recorded as 1; otherwise counts are used (int).
    The build is chunked to keep peak memory low.

    Parameters:
    - dump_dir: directory with per-genome k-mer dump files.
    - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
    - vocab: ordered list of k-mers corresponding to columns in the matrix.
    - binary: whether to collapse counts to binary presence/absence.
    - chunk_size: number of genomes per chunk when building the matrix.

    Returns:
    - scipy.sparse.csr_matrix with dtype `np.int8` for binary (or `np.int32` for counts).
    """
    vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
    dump_dir = Path(dump_dir)
    blocks = []
    dtype = np.int8 if binary else np.int32
    n_features = len(vocab)

    for start in range(0, len(genome_ids), chunk_size):
        chunk_ids = genome_ids[start:start + chunk_size]
        rows: list[int] = []
        cols: list[int] = []
        data: list[int] = []
        for row_idx, gid in enumerate(chunk_ids):
            dump_path = dump_dir / f"{gid}_db_kmers.txt"
            if not dump_path.exists():
                continue
            kmers = iter_genome_kmers(dump_path)
            for kmer in kmers.keys():
                col_idx = vocab_index.get(kmer)
                if col_idx is None:
                    continue
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(1 if binary else int(kmers.get(kmer, 0)))
        if rows:
            block = sparse.csr_matrix((data, (rows, cols)), shape=(len(chunk_ids), n_features), dtype=dtype)
        else:
            block = sparse.csr_matrix((len(chunk_ids), n_features), dtype=dtype)
        blocks.append(block)
    if not blocks:
        return sparse.csr_matrix((0, n_features), dtype=dtype)
    return sparse.vstack(blocks, format='csr')


def load_labels(labels_path: Path) -> pd.Series:
    """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

    Expects a CSV with at least `Genome ID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

    Parameters:
    - labels_path: Path to CSV containing labels.

    Returns:
    - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
    """
    df = pd.read_csv(labels_path)

    # Normalize column lookup so small header variations do not break the notebook.
    norm = {c.strip().lower(): c for c in df.columns}
    gid_col = norm.get("genome id")
    pheno_col = norm.get("phenotype")

    if gid_col is None or pheno_col is None:
        raise ValueError(
            "Expected columns for Genome ID and phenotype in labels file. "
            f"Found columns: {list(df.columns)}"
        )

    pheno = df.set_index(gid_col)[pheno_col]
    pheno = pd.to_numeric(pheno, errors="coerce")
    return pheno

## Run Feature selection and Train models
Adjust the paths below (`dump_dir`, `labels_path`, `genome_ids_path`) if your files are elsewhere, then run this cell.

### Build final binary sparse data and Train model

In [ ]:
# Sample 500 Resistant + 500 Susceptible and build sparse matrix
seed = 42
from pathlib import Path
import random

labels_path = Path('../data/phenotype/cefotaxime_phenotype.csv')
labels = load_labels(labels_path)
labels = labels.dropna()
labels = labels.astype(float)

res_ids = labels[labels == 1.0].index.astype(str).tolist()
sus_ids = labels[labels == 0.0].index.astype(str).tolist()

n_per_class = 1338
if len(res_ids) < n_per_class or len(sus_ids) < n_per_class:
    raise ValueError(f'Not enough genomes to sample: have {len(res_ids)} R, {len(sus_ids)} S')

random.seed(seed)
sampled_res = random.sample(res_ids, n_per_class)
sampled_sus = random.sample(sus_ids, n_per_class)
sampled_ids = sampled_res + sampled_sus

out_ids_path = Path('../data/phenotype/cefotaxime_3000_ids.txt')
out_ids_path.parent.mkdir(parents=True, exist_ok=True)
with out_ids_path.open('w') as fh:
    fh.write('\n'.join(sampled_ids))

print(f'Sampled {len(sampled_ids)} genomes (R={n_per_class}, S={n_per_class}), saved to {out_ids_path}')


Sampled 2908 genomes (R=1454, S=1454), saved to ..\data\sampled_data\trimethoprim_sulfamethoxazole_2908_ids.txt


In [14]:

# Build prevalence and vocabulary (prevalence filter)
dump_dir = Path('../data/counted_kmers')
prevalence = build_prevalence(dump_dir, sampled_ids)
print('Unique k-mers seen:', len(prevalence))


KeyboardInterrupt: 

In [ ]:

# Filter by prevalence (2% - 95%) and cap vocab size if too large
vocab = select_vocab_by_prevalence(prevalence, n_genomes=len(sampled_ids), min_frac=0.02, max_frac=0.95)
print('Vocab after prevalence filter:', len(vocab))


Vocab after prevalence filter: 519266


In [7]:
# Build binary presence sparse matrix (rows ordered as sampled_ids)
X = build_sparse_matrix(dump_dir, sampled_ids, vocab, binary=True, chunk_size=100)
print('Built X shape:', X.shape)

# Build label vector aligned to sampled_ids
# Ensure labels index is string-typed to match sampled_ids
labels_str = labels.copy()
labels_str.index = labels_str.index.astype(str)

missing = [gid for gid in sampled_ids if gid not in labels_str.index]
if missing:
    raise KeyError(f'Some sampled ids are missing in labels: {missing[:5]}... total {len(missing)}')

y = np.array([labels_str.loc[gid] for gid in sampled_ids], dtype=int)
print('Built y shape:', y.shape)

# Save sampled ids and vocab (raw) for reproducibility
fv_dir = Path('../output/feature_selection')
fv_dir.mkdir(parents=True, exist_ok=True)
with (fv_dir / 'cip_3k_vocab_raw.txt').open('w', encoding='utf8') as fh:
    fh.write('\n'.join(vocab))
print('Saved sampled ids and raw vocab.')

Built X shape: (3000, 519266)
Built y shape: (3000,)
Saved sampled ids and raw vocab.


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression

# Lower the initial K-Best to reduce extreme noise before the model sees it
k_features = 20000 

pipeline = Pipeline([
    ('selectk', SelectKBest(chi2, k=min(k_features, X.shape[1]))),
    
    # MaxAbsScaler: This scales data but keeps sparse matrices intact 
    # (unlike StandardScaler which breaks sparsity and crashes memory)
    ('scaler', MaxAbsScaler()), 
    # Compress the redundant k-mer blocks into 300 dense, independent signals
    #('svd', TruncatedSVD(n_components=300, random_state=42)),
    # aggressive regularization (ElasticNet) to force non-informative features to exactly 0
    ('lr', LogisticRegression(
        penalty='elasticnet', 
        l1_ratio=0.5,       # Balances between keeping groups of features (L2) and dropping them (L1)
        C=0.1,              # Strong penalty. Lower values = fewer features kept.
        solver='saga', 
        class_weight='balanced', 
        max_iter=5000,      # Increased to ensure the complex solver converges
        n_jobs=1,
        random_state=42
    ))
])
#define cv globally
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# evaluate
scoring = ['accuracy', 'precision', 'recall', 'f1']
scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)
print(scores)

{'fit_time': array([ 30.85743618, 442.06183267,  31.8319211 ,  28.68024206,
        24.75665259]), 'score_time': array([0.      , 4.567909, 0.      , 0.      , 0.      ]), 'test_accuracy': array([  nan, 0.825,   nan,   nan,   nan]), 'test_precision': array([       nan, 0.80952381,        nan,        nan,        nan]), 'test_recall': array([ nan, 0.85,  nan,  nan,  nan]), 'test_f1': array([       nan, 0.82926829,        nan,        nan,        nan])}


c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
4 fits failed out of a total of 5.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\

In [ ]:
print(f"Mean Accuracy: {np.mean(scores['test_accuracy']):.4f} (+/- {np.std(scores['test_accuracy']):.4f})")
print(f"Mean precision: {np.mean(scores['test_precision']):.4f} (+/- {np.std(scores['test_precision']):.4f})")
print(f"Mean Recall: {np.mean(scores['test_recall']):.4f} (+/- {np.std(scores['test_recall']):.4f})")
print(f"Mean F1: {np.mean(scores['test_f1']):.4f} (+/- {np.std(scores['test_f1']):.4f})")

Mean Accuracy: 0.7087 (+/- 0.0212)
Mean precision: 0.7091 (+/- 0.0222)
Mean Recall: 0.7080 (+/- 0.0225)
Mean F1: 0.7085 (+/- 0.0212)


In [ ]:

# # Save pipeline
# out_models = Path('../output/models')
# joblib.dump(pipeline, out_models+'/lr_pipeline')
# print('Saved model to', out_models)


#### Trying out some hyperparameter tuning on LR model

In [ ]:
# Manually find the correct decision threshold for the model by evaluating the precision-recall curve on the training set.
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# 1. Fit the pipeline on a train/test split to analyze thresholds
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipeline.fit(X_train, y_train)

# 2. Get raw probabilities instead of hard 0/1 predictions
y_prob = pipeline.predict_proba(X_val)[:, 1]

# 3. Test thresholds from 0.3 to 0.7 to find where ALL metrics cross 80%
print(f"{'Threshold':<10}{'Accuracy':<10}{'Precision':<10}{'Recall':<10}{'F1-Score':<10}")
print("-" * 50)

for threshold in np.arange(0.3, 0.7, 0.02):
    y_pred_thresh = (y_prob >= threshold).astype(int)
    
    acc = accuracy_score(y_val, y_pred_thresh)
    prec = precision_score(y_val, y_pred_thresh)
    rec = recall_score(y_val, y_pred_thresh)
    f1 = f1_score(y_val, y_pred_thresh)
    
    print(f"{threshold:.2f}       {acc:.3f}     {prec:.3f}     {rec:.3f}     {f1:.3f}")

Threshold Accuracy  Precision Recall    F1-Score  
--------------------------------------------------
0.30       0.682     0.643     0.817     0.720
0.32       0.687     0.651     0.807     0.720
0.34       0.693     0.661     0.793     0.721
0.36       0.693     0.665     0.780     0.718
0.38       0.692     0.666     0.770     0.714
0.40       0.693     0.673     0.753     0.711
0.42       0.690     0.673     0.740     0.705
0.44       0.692     0.680     0.723     0.701
0.46       0.698     0.693     0.713     0.703
0.48       0.695     0.694     0.697     0.696
0.50       0.700     0.705     0.687     0.696
0.52       0.678     0.703     0.617     0.657
0.54       0.675     0.706     0.600     0.649
0.56       0.673     0.711     0.583     0.641
0.58       0.673     0.720     0.567     0.634
0.60       0.678     0.738     0.553     0.632
0.62       0.675     0.744     0.533     0.621
0.64       0.672     0.751     0.513     0.610
0.66       0.670     0.755     0.503     0.604
0.68 

In [ ]:
# Using GridSearchCV to tune
from sklearn.model_selection import GridSearchCV

# Define a hyperparameter grid around your current setup
param_grid = {
    'selectk__k': [20000, 60000],          # Test if fewer features reduces noise better
    'lr__C': [0.01, 0.1, 1.0],             # Regularization strength
    'lr__l1_ratio': [0.2, 0.5, 0.8]        # Balance between L1 and L2 penalties
}

# Run the search optimizing directly for your goal metric (F1-score)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X, y)

print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best Cross-Validated F1 Score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits


PicklingError: Could not pickle the task to send it to the workers.

In [12]:
# building lightgbm model
import lightgbm as lgb

# Tree-based models don't need scaling, and they handle high dimensions very fast
lgb_pipeline = Pipeline([
    ('selectk', SelectKBest(chi2, k=20000)), # Lower to 20k to keep tree training fast
    ('lgb', lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=6,
        class_weight='balanced',
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

# Convert sparse matrix to float32 for LightGBM compatibility
X_lgb = X.astype(np.float32)
# Evaluate LightGBM
scores_lgb = cross_validate(lgb_pipeline, X_lgb, y, cv=cv, scoring=scoring)
print(scores_lgb)

print("LightGBM Metrics:")
print(f"Mean Accuracy: {np.mean(scores_lgb['test_accuracy']):.4f}")
print(f"Mean Precision: {np.mean(scores_lgb['test_precision']):.4f}")
print(f"Mean Recall: {np.mean(scores_lgb['test_recall']):.4f}")
print(f"Mean F1: {np.mean(scores_lgb['test_f1']):.4f}")


MemoryError: Unable to allocate 769. MiB for an array with shape (201599359,) and data type float32

In [ ]:
# # Train XGBoost
# from xgboost import XGBClassifier

# # 1. XGBoost
# # Note: Use 'scale_pos_weight' if your classes are imbalanced
# xgb_pipeline = Pipeline([
#     ('selectk', SelectKBest(mutual_info_classif, k=min(k_features, X.shape[1]))),
#     # ('svd', TruncatedSVD(n_components=min(svd_components, min(k_features, X.shape[1]) - 1), random_state=seed)),
#     ('xgb', XGBClassifier(
#     n_estimators=100, 
#     learning_rate=0.1, 
#     max_depth=5, 
#     use_label_encoder=False, 
#     eval_metric='logloss'
# ))
# ])
# scores_xgb = cross_validate(xgb_pipeline, X, y, cv=cv, scoring=scoring)
# print(scores_xgb)



### Using Embedded models to select features without SelectKbest

In [ ]:
embeded_pipe = Pipeline([
    ('emb', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, class_weight='balanced', l1_ratio=0.5, C=0.1, random_state=42))
])

scores_emb = cross_validate(embeded_pipe, X, y, cv=cv, scoring=scoring)

In [ ]:
# Extract learned features
all_fold_coefs = []
for train_idx, _ in cv.split(X, y):
    X_fold_train = X[train_idx]
    y_fold_train = y[train_idx]
    
    embeded_pipe.fit(X_fold_train, y_fold_train)
    
    #extract coeffs
    coefs = embeded_pipe.named_steps['emb'].coef_[0]
    all_fold_coefs.append(coefs)
# average importance across all folds
mean_importance = np.mean(np.abs(all_fold_coefs), axis=0)
# get indices of the top 100 most stable kmers
top_kmer_indices =np.argsort(mean_importance)[-100:]

In [ ]:
#Usiing gridsearch to optimize C
from sklearn.model_selection import GridSearchCV

param_grid = {'emb__C': [0.001, 0.01, 0.1, 1, 10]}
grid_search = GridSearchCV(embeded_pipe, param_grid, cv=5, scoring='f1')
grid_search.fit(X,y)
print(f'Best C value; {grid_search.best_params_}')

In [ ]:

# # Save selected k-mers
# selected_idx = lr.named_steps['selectk'].get_support(indices=True)
# selected_kmers = [vocab[i] for i in selected_idx]
# with (fv_dir / 'ampicillin_1000_selected_kmers.txt').open('w', encoding='utf8') as fh:
#     fh.write('\n'.join(selected_kmers))
# print('Saved selected k-mers:', len(selected_kmers))

# Save test predictions (with Genome IDs)
pred_df = pd.DataFrame({
    'GenomeID': ids[test_idx].astype(str),
    'y_true': y_test,
    'y_pred': y_pred,
})
pred_df.to_csv(out_models / 'ampicillin_pilot_test_predictions.csv', index=False)
print('Saved test predictions to', out_models / 'ampicillin_pilot_test_predictions.csv')

## Testing model on Held out dataset

In [ ]:
# Evaluate held-out genomes (not in the 1000 sample)
# Builds features for the held-out genomes, runs the saved model, computes metrics, and saves predictions.
from sklearn.metrics import classification_report, balanced_accuracy_score, average_precision_score, roc_auc_score

labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')
labels_all = load_labels(labels_path).dropna()

# load sampled ids
sampled_path = Path('../data/phenotype/ampicillin_1000_ids.txt')
if sampled_path.exists():
    sampled_ids_file = [s.strip() for s in sampled_path.read_text(encoding='utf8').splitlines() if s.strip()]
else:
    # fall back to in-memory variable if present
    sampled_ids_file = sampled_ids if 'sampled_ids' in globals() else []

# Determine held-out IDs (strings) that have labels
held_ids = [str(g) for g in labels_all.index.astype(str) if str(g) not in set(sampled_ids_file)]
print(f'Total labeled genomes: {len(labels_all)}, held-out candidates: {len(held_ids)}')

# Check for available k-mer dumps and filter
cand_dump_dirs = [Path('../output/counted_kmers'), Path('../data/counted_kmers')]
for d in cand_dump_dirs:
    if d.exists():
        dump_dir = d
        break
else:
    raise FileNotFoundError('Could not find counted_kmers directory in expected locations')

held_ids_with_dump = [gid for gid in held_ids if (dump_dir / f'{gid}_db_kmers.txt').exists()]
print(f'Held-out genomes with dumps: {len(held_ids_with_dump)}')
if not held_ids_with_dump:
    raise RuntimeError('No held-out genome dump files found; cannot evaluate')

# Load saved model
model_path = Path('../output/models/ampicillin_pilot_logreg.joblib')
if not model_path.exists():
    raise FileNotFoundError(f'Model not found at {model_path}')
model = joblib.load(model_path)

# Determine which vocab to use to build features
fv_dir = Path('../output/feature_selection')
raw_vocab_path = fv_dir / 'ampicillin_1000_vocab_raw.txt'
sel_vocab_path = fv_dir / 'ampicillin_1000_selected_kmers.txt'
if raw_vocab_path.exists():
    vocab_for_model = [l.rstrip('\n') for l in raw_vocab_path.read_text(encoding='utf8').splitlines() if l.strip()]
    print('Using raw vocab (pre-selection) with', len(vocab_for_model), 'k-mers')
elif sel_vocab_path.exists():
    vocab_for_model = [l.rstrip('\n') for l in sel_vocab_path.read_text(encoding='utf8').splitlines() if l.strip()]
    print('Using selected k-mers vocab with', len(vocab_for_model), 'k-mers')
else:
    # fallback: attempt to infer from saved model
    if hasattr(model, 'named_steps') and 'selectk' in model.named_steps:
        raise FileNotFoundError('Raw vocab required by pipeline but not found in feature_selection folder')
    else:
        # If model is a plain classifier, attempt to use selected_kmers if present
        vocab_for_model = []

# If we have a vocab, build sparse matrix; otherwise try to error with guidance
if vocab_for_model:
    X_held = build_sparse_matrix(dump_dir, held_ids_with_dump, vocab_for_model, binary=True, chunk_size=100)
else:
    raise RuntimeError('No vocabulary available to construct held-out feature matrix')

# Align labels
y_held = np.array([labels_all.loc[gid] for gid in held_ids_with_dump], dtype=int)

# If model is a pipeline, call predict/predict_proba directly. If it's a bare classifier, ensure feature dims match.
try:
    y_pred = model.predict(X_held)
except Exception as e:
    # If model expects dense input or different shape, try converting
    try:
        y_pred = model.predict(X_held.toarray())
    except Exception:
        raise

try:
    y_proba = model.predict_proba(X_held)[:, 1]
except Exception:
    try:
        y_proba = model.predict_proba(X_held.toarray())[:, 1]
    except Exception:
        y_proba = None

print('Held-out Balanced Accuracy:', balanced_accuracy_score(y_held, y_pred))
print('Held-out Classification Report:\n', classification_report(y_held, y_pred))
if y_proba is not None:
    print('Held-out Average Precision (PR-AUC):', average_precision_score(y_held, y_proba))
    try:
        print('Held-out ROC AUC:', roc_auc_score(y_held, y_proba))
    except Exception:
        pass

# Save predictions
out_models = Path('../output/models')
out_models.mkdir(parents=True, exist_ok=True)
pred_df = pd.DataFrame({
    'GenomeID': held_ids_with_dump,
    'y_true': y_held,
    'y_pred': y_pred,
})
if y_proba is not None:
    pred_df['y_proba'] = y_proba

pred_file = out_models / 'ampicillin_heldout_predictions.csv'
pred_df.to_csv(pred_file, index=False)
print('Saved held-out predictions to', pred_file)

# Save a short summary
summary = {
    'held_count': int(len(held_ids_with_dump)),
    'balanced_accuracy': float(balanced_accuracy_score(y_held, y_pred)),
}
with open(out_models / 'ampicillin_heldout_summary.json', 'w') as fh:
    json.dump(summary, fh)
print('Saved held-out summary')